# Biomedical relationship demo

This notebook shows a schema-free first-pass pipeline for comparing messy biomedical records.

It: (1) parses mixed text fields, (2) extracts typed biological mentions, (3) keeps contextual text, and (4) scores relatedness between two records.

In [ ]:
from __future__ import annotations

import json
import math
import re
from collections import defaultdict
from dataclasses import dataclass
from difflib import SequenceMatcher
from functools import lru_cache
from pprint import pprint
from typing import Any, Dict, Iterable, List, Optional, Tuple
from urllib.parse import quote_plus

import requests
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

BASE_STOPWORDS = set(ENGLISH_STOP_WORDS)

In [20]:
record_1 = json.loads(r'''
{
  "relevant_experimental_therapeutic_intervention": "[]",
  "created": "2026-02-26T19:28:28.409000000Z",
  "relevant_human_pathways": "[\"epithelial-mesenchymal transition pathways in sarcomatoid subtype bladder cancer\"]",
  "relevant_human_cancer": "[\"Bladder Cancer\"]",
  "human_relevance_record_id": "ORGANOIDS01",
  "uuid": "b92fe2a6-d695-5851-a4fc-5fa1772060c8",
  "human_relevance_statement": "Muscle invasive bladder cancer in humans is a heterogeneous disease that is very difficult to treat and is associated with poor survival. Bladder cancer in dogs is a spontaneously occurring cancer that closely resembles human muscle invasive bladder cancer and has been demonstrated to be a great pre-clinical model for the disease. This study is the first that developed 3-dimensional cancer cell lines derived from dogs with bladder cancer which will be useful to screen possible novel drugs before taking them into clinical trials in dogs and subsequently, people with bladder cancer. With this study, we have shown that canine bladder cancer organoids closely represent their parent tumors. Importantly, we also show that the canine bladder cancer organoids match important carcinogenesis pathways of human bladder cancer, including basal and luminal molecular subtypes, as shown in transcriptomic pathway upregulations (such as epithelial-mesenchymal transition pathways in the most aggressive and invasive tumors), and important somatic mutations such as p53, PIK3CA, KDM6A and KRAS.",
  "type_": "human_relevance",
  "relevant_human_genes": "[\"p53, PIK3CA, KDM6A and KRAS\"]",
  "nci_link_to_relevant_human_cancer": "https://www.cancer.gov/types/bladder"
}
''')

record_2 = json.loads(r'''
{
  "relevant_experimental_therapeutic_intervention": "[\"immunotherapies, remodeling the TME\"]",
  "created": "2026-02-26T19:28:28.418000000Z",
  "relevant_human_pathways": "[\"Transcriptional regulation by RUNX2, transcriptional regulation by TP53, Transcriptional regulation of pluripotent stem cells\"]",
  "relevant_human_cancer": "[\"Osteosarcoma\"]",
  "human_relevance_record_id": "OSA03",
  "uuid": "20bf9722-c3d9-52f4-908a-50ae2b3a5a05",
  "human_relevance_statement": "This study allowed us to identify how osteosarcomas from dogs and humans use gene methylation, an important epigenetic mechanism, to achieve comparable biological behavior despite their vastly different mutational landscapes. Furthermore, it enabled us to uncover how the tumors in both species can evolve to convergent phenotypes using genetic changes in one species and epigenetic changes in the other.",
  "type_": "human_relevance",
  "relevant_human_genes": "[\"TP53, PTEN, RB, MYC, RUNX2\"]",
  "nci_link_to_relevant_human_cancer": "https://www.cancer.gov/types/bone"
}
''')


In [21]:

# ---------- Parsing helpers ----------


def parse_listish(value: Any) -> List[str]:
    if value is None:
        return []
    if isinstance(value, list):
        return [str(x).strip() for x in value if str(x).strip()]
    if not isinstance(value, str):
        return [str(value).strip()] if str(value).strip() else []

    s = value.strip()
    if not s:
        return []

    try:
        parsed = json.loads(s)
        if isinstance(parsed, list):
            return [str(x).strip() for x in parsed if str(x).strip()]
        if isinstance(parsed, str) and parsed.strip():
            return [parsed.strip()]
    except Exception:
        pass

    return [s]


def looks_like_metadata_key(key: str) -> bool:
    key_l = key.lower()
    return bool(
        re.search(r"(^|_)(uuid|created|updated|timestamp|type|record_id|id|url|link)$", key_l)
        or key_l.endswith("_id")
    )


def looks_like_metadata_value(value: str) -> bool:
    s = value.strip()
    if not s:
        return True
    if s.startswith("http://") or s.startswith("https://"):
        return True
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}t\d{2}:\d{2}:\d{2}.*z?", s.lower()):
        return True
    return False


def build_raw_text(record: Dict[str, Any]) -> str:
    parts: List[str] = []

    for key, value in record.items():
        if looks_like_metadata_key(str(key)):
            continue

        for item in parse_listish(value):
            item = item.strip()
            if item and not looks_like_metadata_value(item):
                parts.append(item)

    return " ".join(parts)

In [22]:
raw_text_1 = build_raw_text(record_1)
raw_text_1


'epithelial-mesenchymal transition pathways in sarcomatoid subtype bladder cancer Bladder Cancer Muscle invasive bladder cancer in humans is a heterogeneous disease that is very difficult to treat and is associated with poor survival. Bladder cancer in dogs is a spontaneously occurring cancer that closely resembles human muscle invasive bladder cancer and has been demonstrated to be a great pre-clinical model for the disease. This study is the first that developed 3-dimensional cancer cell lines derived from dogs with bladder cancer which will be useful to screen possible novel drugs before taking them into clinical trials in dogs and subsequently, people with bladder cancer. With this study, we have shown that canine bladder cancer organoids closely represent their parent tumors. Importantly, we also show that the canine bladder cancer organoids match important carcinogenesis pathways of human bladder cancer, including basal and luminal molecular subtypes, as shown in transcriptomic p

In [23]:
raw_text_2 = build_raw_text(record_2)
raw_text_2

'immunotherapies, remodeling the TME Transcriptional regulation by RUNX2, transcriptional regulation by TP53, Transcriptional regulation of pluripotent stem cells Osteosarcoma This study allowed us to identify how osteosarcomas from dogs and humans use gene methylation, an important epigenetic mechanism, to achieve comparable biological behavior despite their vastly different mutational landscapes. Furthermore, it enabled us to uncover how the tumors in both species can evolve to convergent phenotypes using genetic changes in one species and epigenetic changes in the other. human_relevance TP53, PTEN, RB, MYC, RUNX2'

In [24]:
def build_corpus(records):
    return [build_raw_text(r) for r in records]

def get_dynamic_stop_terms(texts, doc_freq_threshold=0.85, min_term_len=3):
    """
    Terms appearing in most documents behave like stopwords in this corpus.
    """
    vectorizer = TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=1
    )
    X = vectorizer.fit_transform(texts)
    terms = vectorizer.get_feature_names_out()
    df = (X > 0).sum(axis=0).A1
    n_docs = X.shape[0]

    stop_terms = [
        term for term, freq in zip(terms, df)
        if (freq / n_docs) >= doc_freq_threshold and len(term) >= min_term_len
    ]
    return stop_terms, vectorizer

In [26]:
def filter_text_with_stop_terms(text, stop_terms):
    stop_set = set(stop_terms)
    tokens = re.findall(r"[A-Za-z0-9\-]+", text.lower())
    return " ".join([t for t in tokens if t not in stop_set])

def build_filtered_corpus(records, stop_terms):
    return [filter_text_with_stop_terms(build_raw_text(r), stop_terms) for r in records]

In [27]:
records = [record_1, record_2]   # or your full dataset
texts = build_corpus(records)

stop_terms, fitted_vectorizer = get_dynamic_stop_terms(texts, doc_freq_threshold=0.85)
filtered_texts = build_filtered_corpus(records, stop_terms)

print("Dynamic stop terms:", stop_terms)
print("Record 1 filtered:", filtered_texts[0])
print("Record 2 filtered:", filtered_texts[1])

Dynamic stop terms: ['and', 'dogs', 'dogs and', 'from', 'from dogs', 'human_relevance', 'humans', 'important', 'in the', 'study', 'the', 'their', 'this', 'this study', 'tumors']
Record 1 filtered: epithelial-mesenchymal transition pathways in sarcomatoid subtype bladder cancer bladder cancer muscle invasive bladder cancer in is a heterogeneous disease that is very difficult to treat is associated with poor survival bladder cancer in is a spontaneously occurring cancer that closely resembles human muscle invasive bladder cancer has been demonstrated to be a great pre-clinical model for disease is first that developed 3-dimensional cancer cell lines derived with bladder cancer which will be useful to screen possible novel drugs before taking them into clinical trials in subsequently people with bladder cancer with we have shown that canine bladder cancer organoids closely represent parent importantly we also show that canine bladder cancer organoids match carcinogenesis pathways of human